# AI-Powered Plant Identification & Geometry Correction

This notebook is the **AI enrichment step** in the field collection workflow. It uses the **Collector view layer** to:

1. **Query** features with `review_status = 'Pending Review'` that have photo attachments
2. **Analyse** each photo via `analyze_image` (ArcGIS AI Utility Services) to extract:
   - Common name, Latin name, Family, Plant type, Condition, Height (m)
3. **Correct geometry** — if `canopy_width_m` is available, replace the feature geometry with a geodesic circle centered on the feature's centroid
4. **Set status** to `In Review` so the reviewer can QA/QC the AI results in the In Review web map

> **Input**: Collector view layer Item ID (from the `manage-views-and-webmaps` notebook)
> **Output**: AI-enriched features with corrected circle geometries, status = "In Review"

In [22]:
import getpass, urllib3
from collections import Counter
from statistics import median

from arcgis.gis import GIS
from arcgis.ai import analyze_image, analyze_text, AIUtilsResponse, AIUtilsException
from arcgis.features import Feature, FeatureLayerCollection

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Staging toggle ─────────────────────────────────────────────────────
# Set to True to use the staging registry (item_registry_staging.json).
STAGING = False
# ───────────────────────────────────────────────────────────────────────

## Step 1 – Connect to ArcGIS Online

In [23]:
username = input("Username: ").strip()
password = getpass.getpass("Password: ")

gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Logged in as {gis.users.me.username} ({gis.url})")

Setting `verify_cert` to False is a security risk, use at your own risk.


Logged in as ralouta.aiddev (https://www.arcgis.com)


## Step 2 – Load the Collector View Layer

Auto-loads the **Collector view** item ID from `item_registry.json` (created by the `manage-views-and-webmaps` notebook). The AI enrichment queries only features with `review_status = 'Pending Review'`. Falls back to manual input if the registry is not found.

In [24]:
import json
from pathlib import Path

if STAGING:
    _registry_path = Path("..") / "item_registry_staging.json"
    print("⚠️  STAGING MODE")
else:
    _registry_path = Path("..") / "item_registry.json"

item_id = None

if _registry_path.exists():
    _reg = json.loads(_registry_path.read_text())
    item_id = _reg.get("views", {}).get("collector", {}).get("item_id")
    if item_id:
        print(f"Auto-loaded Collector view ID from {_registry_path.name}: {item_id}")

if not item_id:
    item_id = input("Collector view layer Item ID: ").strip()

fl_item = gis.content.get(item_id)
feature_layer = fl_item.layers[0]

total_count = feature_layer.query(return_count_only=True)
pending_count = feature_layer.query(where="review_status = 'Pending Review'", return_count_only=True)

print(f"Total features: {total_count}")
print(f"Pending Review: {pending_count}")

Auto-loaded Collector view ID from item_registry.json: bc3f721e9c514d36b38947f23e1c0079
Total features: 3183
Pending Review: 1


## Step 3 – Query Pending Features with Attachments

Fetch features where `review_status = 'Pending Review'`, then filter to only those that have at least one photo attachment.

In [25]:
oid_field = feature_layer.properties.objectIdField

# check if attachments are enabled on the view
has_att = getattr(feature_layer.properties, "hasAttachments", False)
print(f"hasAttachments on view: {has_att}")
if not has_att:
    print("⚠ Enabling attachments on this view layer...")
    feature_layer.manager.update_definition({"hasAttachments": True})
    print("  Done.")

# 1. lightweight query — OID field only, no geometry
pending_fs = feature_layer.query(
    where="review_status = 'Pending Review'",
    out_fields=oid_field,
    return_geometry=False,
).features
pending_oids = [f.attributes[oid_field] for f in pending_fs]
print(f"{len(pending_oids)} pending features (OIDs: {pending_oids})")

# 2. filter to OIDs that have at least one attachment
oids_with_att = {}  # {oid: att_list}
for oid in pending_oids:
    try:
        att_list = feature_layer.attachments.get_list(oid)
    except Exception:
        att_list = []
    if att_list:
        oids_with_att[oid] = att_list

print(f"{len(oids_with_att)} of those have attachments")

# 3. full query only for the features we'll actually process
if oids_with_att:
    oid_csv = ",".join(str(o) for o in oids_with_att)
    features = feature_layer.query(
        where=f"{oid_field} IN ({oid_csv})",
        out_fields="*",
        return_geometry=True,
    ).features
    for f in features:
        f._att_list = oids_with_att[f.attributes[oid_field]]
else:
    features = []

print(f"{len(features)} feature(s) to process")

hasAttachments on view: True
1 pending features (OIDs: [3175])
1 of those have attachments
1 feature(s) to process


## Step 4 – Define AI Prompts

Each prompt maps to a field in the tree schema. The AI will analyse the plant photo and return a value for each field. Domain-constrained fields (`plant_type`, `condition`) include the allowed values in the prompt so the AI returns a valid code.

In [26]:
prompt_data = [
    {
        "key": "shot_type",
        "context": (
            "Classify this photo as exactly ONE of: 'closeup' or 'wide'. "
            "'closeup' = zoomed-in view of leaves, flowers, bark, fruit, or trunk detail "
            "(good for species ID but NOT for measuring size). "
            "'wide' = full plant visible from a distance, showing overall shape, height, and canopy spread "
            "(good for measuring height/width but NOT for species ID). "
            "Return ONLY one word: closeup or wide."
        ),
    },
    {
        "key": "common_name",
        "context": (
            "Identify the main plant in this photo. "
            "Return ONLY its common English name (e.g. 'Coast Live Oak')."
        ),
    },
    {
        "key": "latin_name",
        "context": (
            "Identify the main plant in this photo. "
            "Return ONLY its binomial Latin/scientific name (e.g. 'Quercus agrifolia')."
        ),
    },
    {
        "key": "family",
        "context": (
            "Identify the main plant in this photo. "
            "Return ONLY its botanical family name (e.g. 'Fagaceae')."
        ),
    },
    {
        "key": "plant_type",
        "context": (
            "Classify the main plant in this photo into exactly ONE of these categories: "
            "Tree, Shrub, Herb, Vine, Succulent, Grass. "
            "Return ONLY one word from that list."
        ),
    },
    {
        "key": "condition",
        "context": (
            "Assess the overall health/condition of the main plant in this photo. "
            "Choose exactly ONE of: Healthy, Stressed, Diseased, Dead. "
            "Return ONLY one word from that list."
        ),
    },
    {
        "key": "height_m",
        "context": (
            "Estimate the height of the main plant in this photo in meters. "
            "Return ONLY a decimal number (e.g. '4.5')."
        ),
    },
    {
        "key": "canopy_width_m",
        "context": (
            "Estimate the canopy width (diameter of the crown spread) of the main plant "
            "in this photo in meters. Return ONLY a decimal number (e.g. '3.0')."
        ),
    },
]


## Step 5 – Analyse Photos and Collect Updates

Two-phase AI analysis for each feature:

**Phase 1 — `analyze_image`**: Each photo attachment (full plant, leaf close-up, trunk, etc.) is analysed independently. The AI also classifies each photo as `closeup` or `wide`.

**Phase 2 — shot-type aware merge**: When multiple photos exist:
- **ID fields** (`common_name`, `latin_name`, `family`, `plant_type`, `condition`) prefer values from **closeup** photos (leaves/bark/flowers give better species ID).
- **Size fields** (`height_m`, `canopy_width_m`) prefer values from **wide** photos (full plant visible).
- If no shot of the preferred type exists, fall back to the other type.
- If remaining values still conflict, `analyze_text` reconciles across the evidence; if that fails, a majority-vote / median fallback is used.

Finally, `review_status` is set to `In Review`.


In [27]:
NUMERIC_FIELDS = {"height_m", "canopy_width_m"}
IMAGE_CONTENT_TYPES = {"image/jpeg", "image/png", "image/gif", "image/webp", "image/tiff"}

# Fields that should be sourced from CLOSEUP photos (species ID works best on detail shots)
ID_FIELDS = {"common_name", "latin_name", "family", "plant_type", "condition"}
# Fields that should be sourced from WIDE photos (size estimates need the full plant visible)
SIZE_FIELDS = {"height_m", "canopy_width_m"}
# Fields that go into the final output (shot_type is metadata only)
OUTPUT_FIELDS = ID_FIELDS | SIZE_FIELDS

# Build reconciliation prompts from prompt_data — no duplication.
# Skip shot_type: reconciliation works on already-filtered results per field type.
_RECONCILE_PREFIX = (
    "Based on ALL the photo analyses above (same plant, different angles/parts), "
    "determine the single most accurate answer. "
)
reconcile_prompts = [
    {"key": p["key"], "context": _RECONCILE_PREFIX + p["context"]}
    for p in prompt_data if p["key"] != "shot_type"
]


def _has_conflicts(all_results: list[dict], fields: set[str] | None = None) -> bool:
    """Return True if any field (optionally filtered) has differing values across results."""
    keys = {k for res in all_results for k in res} if fields is None else fields
    for key in keys:
        values = [res.get(key) for res in all_results if res.get(key) is not None]
        if len(set(str(v) for v in values)) > 1:
            return True
    return False


def _parse_ai_results(results) -> dict:
    """Parse AI response results into an attribute dict."""
    attrs = {}
    for r in results:
        val = r.value.strip() if isinstance(r.value, str) else r.value
        if r.key in NUMERIC_FIELDS:
            try:
                val = float(val)
            except (ValueError, TypeError):
                val = None
        attrs[r.key] = val
    return attrs


def _reconcile_with_ai(all_results: list[dict], att_names: list[str]) -> dict:
    """Use analyze_text to reconcile conflicting per-image results."""
    lines = [
        "Multiple photos of the SAME plant were taken from different angles or parts "
        "(e.g. whole plant, leaves, trunk, flowers). Each photo was analyzed independently. "
        "The results may differ because close-up photos show different features. "
        "Determine the single most accurate value for each field by reasoning across all photos.\n"
    ]
    for name, res in zip(att_names, all_results):
        parts = ", ".join(f"{k}={v}" for k, v in res.items() if v is not None)
        lines.append(f"- {name}: {parts}")

    response = analyze_text(text="\n".join(lines), prompt=reconcile_prompts)

    if isinstance(response, AIUtilsException):
        print(f"  ⚠ Reconciliation failed: {response.value}")
        return {}

    return _parse_ai_results(response.results)


def _simple_merge(all_results: list[dict], fields: set[str] | None = None) -> dict:
    """Fallback merge: majority vote for strings, median for numerics.

    If `fields` is given, only those keys are merged.
    """
    merged = {}
    per_key: dict[str, list] = {}
    for res in all_results:
        for k, v in res.items():
            if fields is not None and k not in fields:
                continue
            per_key.setdefault(k, []).append(v)

    for key, values in per_key.items():
        if key in NUMERIC_FIELDS:
            nums = [v for v in values if v is not None]
            merged[key] = round(median(nums), 2) if nums else None
        else:
            valid = [v for v in values if v]
            merged[key] = Counter(valid).most_common(1)[0][0] if valid else None
    return merged


def _merge_by_shot_type(all_results: list[dict], att_names: list[str]) -> dict:
    """Shot-type-aware merge.

    - ID fields prefer closeup photos; fall back to wide if no closeup exists.
    - Size fields prefer wide photos; fall back to closeup if no wide exists.
    - If the preferred subset still has conflicts, try AI reconciliation then majority/median.
    """
    closeups = [r for r in all_results if str(r.get("shot_type", "")).lower() == "closeup"]
    wides    = [r for r in all_results if str(r.get("shot_type", "")).lower() == "wide"]

    id_pool   = closeups or wides or all_results
    size_pool = wides or closeups or all_results

    print(f"  Shot types → closeup: {len(closeups)}, wide: {len(wides)}, other: {len(all_results) - len(closeups) - len(wides)}")
    print(f"    ID fields from {'closeups' if closeups else ('wides' if wides else 'all')} ({len(id_pool)})")
    print(f"    Size fields from {'wides' if wides else ('closeups' if closeups else 'all')} ({len(size_pool)})")

    merged: dict = {}

    # ── ID fields ─────────────────────────────────────────────────
    if len(id_pool) == 1 or not _has_conflicts(id_pool, ID_FIELDS):
        for k in ID_FIELDS:
            v = next((r.get(k) for r in id_pool if r.get(k) is not None), None)
            merged[k] = v
    else:
        merged.update(_simple_merge(id_pool, ID_FIELDS))

    # ── Size fields ───────────────────────────────────────────────
    if len(size_pool) == 1 or not _has_conflicts(size_pool, SIZE_FIELDS):
        for k in SIZE_FIELDS:
            v = next((r.get(k) for r in size_pool if r.get(k) is not None), None)
            merged[k] = v
    else:
        merged.update(_simple_merge(size_pool, SIZE_FIELDS))

    return merged


# ── Main processing loop ─────────────────────────────────────────────────
# Builds attribute updates only — geometry is already a point (from seed/QA/QC)
# and does not need correction.

updates = []
results_log = {}

for i, feat in enumerate(features, 1):
    oid = feat.attributes[oid_field]

    # Filter to image attachments only
    image_atts = [
        a for a in feat._att_list
        if a.get("contentType", "").lower() in IMAGE_CONTENT_TYPES
    ]
    if not image_atts:
        print(f"[{i}/{len(features)}] OID {oid} – no image attachments, skipping.")
        continue

    print(f"[{i}/{len(features)}] OID {oid} – analysing {len(image_atts)} attachment(s)...")

    # ── Phase 1: analyze each image independently ────────────────────
    per_attachment_results = []
    att_names = []
    for j, att in enumerate(image_atts, 1):
        att_id = att["id"]
        att_name = att.get("name", f"attachment_{att_id}")
        img_url = f"{feature_layer.url}/{oid}/attachments/{att_id}?token={gis._con.token}"

        try:
            response = analyze_image(img_url, prompt_data)
        except Exception as exc:
            print(f"  [{j}/{len(image_atts)}] {att_name} – ERROR: {exc}")
            continue

        if isinstance(response, AIUtilsException):
            print(f"  [{j}/{len(image_atts)}] {att_name} – AI ERROR: {response.value}")
            continue

        att_attrs = _parse_ai_results(response.results)
        if att_attrs:
            per_attachment_results.append(att_attrs)
            att_names.append(att_name)
            print(f"  [{j}/{len(image_atts)}] {att_name} [{att_attrs.get('shot_type', '?')}] → {att_attrs}")
        else:
            print(f"  [{j}/{len(image_atts)}] {att_name} – no results returned")

    if not per_attachment_results:
        print(f"  No usable results from any attachment, skipping.")
        continue

    # ── Phase 2: shot-type-aware merge ────────────────────────────────
    if len(per_attachment_results) == 1:
        only = per_attachment_results[0]
        new_attrs = {k: only.get(k) for k in OUTPUT_FIELDS}
    else:
        new_attrs = _merge_by_shot_type(per_attachment_results, att_names)
        print(f"  Merged → {new_attrs}")

    # Keep only output fields (drop shot_type) then set status
    new_attrs = {k: v for k, v in new_attrs.items() if k in OUTPUT_FIELDS}
    new_attrs["review_status"] = "In Review"
    results_log[oid] = new_attrs

    updates.append({"attributes": {oid_field: oid, **new_attrs}})

print(f"\n{len(updates)} feature(s) ready for attribute update.")


[1/1] OID 3175 – analysing 1 attachment(s)...
  [1/1] Photo 1.jpg [wide] → {'shot_type': 'wide', 'common_name': 'Oak', 'latin_name': 'Quercus', 'family': 'Fagaceae', 'plant_type': 'Tree', 'condition': 'Healthy', 'height_m': 14.0, 'canopy_width_m': 8.0}

1 feature(s) ready for attribute update.


In [28]:
results_log

{3175: {'condition': 'Healthy',
  'canopy_width_m': 8.0,
  'common_name': 'Oak',
  'family': 'Fagaceae',
  'plant_type': 'Tree',
  'height_m': 14.0,
  'latin_name': 'Quercus',
  'review_status': 'In Review'}}

## Step 6 – Apply Attribute Updates

Write AI-enriched attributes (including `review_status = 'In Review'`) back to the layer. Geometry is already a point and does not need correction.

In [29]:
if updates:
    result = feature_layer.edit_features(updates=updates)
    successes = sum(1 for r in result.get("updateResults", []) if r.get("success"))
    print(f"Updated {successes}/{len(updates)} feature(s)")
    # Show any failures
    for r in result.get("updateResults", []):
        if not r.get("success"):
            print(f"  FAILED OID {r.get('objectId')}: {r.get('error', {})}")
else:
    print("No updates to apply.")

Updated 1/1 feature(s)


## Summary

All pending features with attachments have been:
- **AI-analysed**: common name, Latin name, family, plant type, condition, and height extracted from photos
- **Status updated**: set to `In Review` — features move from the Pending Review view to the In Review view

Geometry is left as-is (point features represent the trunk base from the QA/QC step).

Open the **In Review web map** to approve, reject, or send back the AI results.